In [ ]:
# Paths (added for the public repo). Paths below are relative to this notebook;
# override with the DATA_ROOT / RUNS_ROOT environment variables.
import os
DATA_ROOT = os.environ.get('DATA_ROOT', '../../data')   # data/blouse, data/aihub_official
RUNS_ROOT = os.environ.get('RUNS_ROOT', '../../runs')   # checkpoints / metrics are written here


# 폐의류 재활용 분류 — **B2-MS + 개선 (Multi-Scale Texture + Focal/정규화)** (로컬)

핵심 질문: **설명가능성 1등(B2-MS)** 구조에 **분류 개선 레시피(Focal·정규화·임계값)** 를 결합하면
정확도와 설명가능성을 **동시에** 얻어 트레이드오프를 해소할 수 있는가?

**구성 (B2-MS 구조 + 개선판 레시피)**
- 백본/질감분기: B2-MS 와 동일 (멀티스케일 텍스처 + 학습형 α)
- ★ Dropout 헤드 + Focal Loss(γ=2) + RandomErasing + weight_decay↑(5e-4) + epoch↓(15) + 폐기 임계값
- 데이터·split: B0 와 동일(`_split` CSV 재사용) → 공정 비교

> 비교: B2-MS(정확도 0.819 / IoU 최고) vs 개선판(정확도 0.832 / IoU 최저) vs **본 모델**


## 0. 환경 설치

In [ ]:
# Colab 기본 환경에는 대부분 설치되어 있으나, FLOPs 측정용 thop 만 추가 설치
!pip -q install thop seaborn scikit-learn


## 1. Imports & 재현성 시드

In [ ]:
import os, json, random, time, math, copy
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # 속도 우선
    torch.backends.cudnn.benchmark = True

seed_everything(42)


# === [TRUNCATED IMAGE FIX] 잘린 이미지도 로드 ===
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


## 2. 설정 (Config)

실험 변수는 여기에만 두어 Ablation 비교를 용이하게 합니다.

In [ ]:
@dataclass
class Config:
    # 경로 — 본인 환경에 맞게 수정
    # ⚠️ 위에서 find 명령으로 찾은 '01.원천데이터'/'02.라벨링데이터'의 공통 부모 폴더
    data_root: str = f'{DATA_ROOT}/blouse'#drive 내에 있는 데이터
    output_dir: str = f'{RUNS_ROOT}/B2MS_improved'#colab에서 Drive에 결과값 넣는

    # 클래스
    main_classes: tuple = ('재사용', '재활용', '폐기')          # 3-class
    # [MULTI-TASK] sub_classes 는 셀 4에서 데이터로부터 자동 발견됩니다.

    # 학습
    img_size: int = 224
    batch_size: int = 32          # T4 16GB 안정값. OOM 시 16 으로 낮추세요.
    num_workers: int = 0          # Colab 기본은 2가 안전
    epochs: int = 15
    lr: float = 1e-4
    weight_decay: float = 5e-4
    label_smoothing: float = 0.05
    use_amp: bool = True

    # [B1] Texture-aware Block
    use_texture_block: bool = True   # False 로 두면 B0 와 동일 (ablation 토글)
    texture_mid_ch: int = 32

    # 분할 비율 (이미 split 되어 있다면 사용 안 함)
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    seed: int = 42

    # === [개선 레시피] ===
    use_focal: bool = True
    focal_gamma: float = 2.0
    dropout_p: float = 0.3
    use_random_erasing: bool = True
    use_mixup: bool = False
    mixup_alpha: float = 0.2
    tune_dispose_threshold: bool = True

cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

## 3. 데이터 준비 — 매니페스트 빌드

**확인된 AI-Hub JSON 스키마**

```json
{
  "meta_information": {
    "big_category":    "dispose | recycle | reusable",   ← Main label (3-class)
    "medium_category": "blouse | ...",                    ← Sub label
    "file_name":       "DP_BL_00085.jpg",                 ← 매칭 이미지
    ...
  },
  "annotations": {
    "clothes":    [{"type": ..., "bbox": [...]}],        ← 의류 bbox
    "pollution":  [{"type": ..., "bbox": [...]}],        ← 오염 bbox (XAI GT!)
    "damage":     [{"type": ..., "bbox": [...]}],        ← 손상 bbox (XAI GT!)
    "attachment": [...]
  }
}
```

파일명 prefix(`DP_`/`RC_`/`RU_`)로도 Main 라벨을 알 수 있어 fallback 으로 사용합니다.

`pollution` / `damage` bbox 는 **XAI 평가의 GT** 입니다 — §4 의 WSOL 기반
정밀도 평가에서 Grad-CAM 히트맵과 IoU 비교에 그대로 활용됩니다.
지금 단계에선 manifest 에 같이 저장만 해두고, B4 단계에서 사용합니다.

In [ ]:
# (로컬 실행) Drive 마운트 불필요.
print('local run: drive mount skipped')


In [ ]:
import json
import pandas as pd
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# AI-Hub 실제 스키마 (사용자 업로드 샘플로 확인됨)
STATE_MAP = {
    'dispose':  '폐기',
    'recycle':  '재활용',
    'reusable': '재사용',
}
# 파일명 prefix → 상태 (JSON 누락 시 fallback)
PREFIX_MAP = {'DP': '폐기', 'RC': '재활용', 'RU': '재사용'}


def state_from_filename(name: str):
    pref = Path(name).stem.split('_')[0].upper()
    return PREFIX_MAP.get(pref)


def parse_label_json(json_path: Path, image_lookup: dict):
    """AI-Hub JSON → dict 또는 None.

    image_lookup: {파일명(stem) → 절대경로} 형태의 사전 빌드 인덱스 (빠른 매칭).
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception:
        return None

    meta = data.get('meta_information', {})
    big  = meta.get('big_category', '').strip().lower()
    med  = meta.get('medium_category', 'unknown').strip().lower()
    fname = meta.get('file_name') or f'{json_path.stem}.jpg'

    main = STATE_MAP.get(big) or state_from_filename(fname)
    if main is None:
        return None

    # 이미지 경로 찾기 (인덱스 우선, 없으면 stem 매칭)
    img_path = image_lookup.get(fname) or image_lookup.get(Path(fname).stem)
    if img_path is None:
        return None

    ann = data.get('annotations', {})
    return {
        'file_path': str(img_path),
        'main': main,
        'sub':  med,
        # XAI 단계에서 사용할 bbox 들 (없으면 빈 리스트)
        'clothes_bbox':    [b['bbox'] for b in ann.get('clothes', [])],
        'pollution_bbox':  [b['bbox'] for b in ann.get('pollution', [])],
        'damage_bbox':     [b['bbox'] for b in ann.get('damage', [])],
        'attachment_bbox': [b['bbox'] for b in ann.get('attachment', [])],
        'caption': data.get('image_caption', {}).get('caption', ''),
    }


def build_manifest(data_root: str) -> pd.DataFrame:
    """data_root 아래의 모든 JSON 을 훑어 매니페스트 DataFrame 을 만든다.

    폴더 이름이 무엇이든 (원천데이터/라벨링데이터, images/labels, 또는 그냥 섞여있어도)
    재귀 탐색으로 동작한다.
    """
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"data_root not found: {data_root}")

    # 1) 이미지 인덱스 빌드 (한 번만 rglob)
    print('Indexing images...', flush=True)
    image_lookup = {}
    for p in data_root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            image_lookup[p.name] = p                  # 파일명 전체
            image_lookup.setdefault(p.stem, p)        # stem only
    print(f'  {len(image_lookup)} image files indexed')

    # 2) JSON 순회
    print('Parsing JSON labels...', flush=True)
    rows, skipped = [], 0
    for jp in data_root.rglob('*.json'):
        r = parse_label_json(jp, image_lookup)
        if r is None:
            skipped += 1
            continue
        rows.append(r)
    df = pd.DataFrame(rows).drop_duplicates(subset='file_path').reset_index(drop=True)
    print(f'  parsed {len(df)} samples, skipped {skipped}')
    return df


df = build_manifest(cfg.data_root)
assert len(df) > 0, '매니페스트가 비었습니다. cfg.data_root 경로 확인 필요.'
display(df.drop(columns=[c for c in df.columns if c.endswith('_bbox')]).head())
print('\n[Main] 클래스 분포'); print(df['main'].value_counts())
print('\n[Sub]  클래스 분포 (상위 15)'); print(df['sub'].value_counts().head(15))
print('\n오염 bbox 보유 샘플:', (df['pollution_bbox'].apply(len) > 0).sum())
print('손상 bbox 보유 샘플:', (df['damage_bbox'].apply(len) > 0).sum())

# ---------------------------------------------------------
# [수정된 부분] 매니페스트 저장 전 폴더 생성 로직 추가
# ---------------------------------------------------------
manifest_path = Path(cfg.output_dir) / 'manifest.parquet'

# 부모 디렉토리(cfg.output_dir)가 없으면 자동으로 생성합니다.
manifest_path.parent.mkdir(parents=True, exist_ok=True)

try:
    df.to_parquet(manifest_path)
    print('\nSaved manifest →', manifest_path)
except Exception:
    manifest_path = Path(cfg.output_dir) / 'manifest.csv'
    df.to_csv(manifest_path, index=False)
    print('\nSaved manifest (csv) →', manifest_path)

In [ ]:
# (로컬 실행) Drive 마운트 불필요.
print('local run: drive mount skipped')


## 4. Train / Val / Test 분할 (Stratified)

In [ ]:
# === [LOCAL FAIR-SPLIT] B0 split 재사용 (공정 비교) ===
df = df[df['main'].isin(cfg.main_classes)].reset_index(drop=True)
main2idx = {c: i for i, c in enumerate(cfg.main_classes)}
df['main_idx'] = df['main'].map(main2idx)
SUB_CLASSES = tuple(df['sub'].value_counts().index.tolist())
df['sub_idx'] = df['sub'].map({c:i for i,c in enumerate(SUB_CLASSES)})
import os as _os
_split_dir = f'{RUNS_ROOT}/_split'
_tef=_os.path.join(_split_dir,'test_files.csv')
if _os.path.exists(_tef):
    _tr=set(pd.read_csv(_os.path.join(_split_dir,'train_files.csv'))['file_path'])
    _va=set(pd.read_csv(_os.path.join(_split_dir,'val_files.csv'))['file_path'])
    _te=set(pd.read_csv(_tef)['file_path'])
    train_df=df[df['file_path'].isin(_tr)].reset_index(drop=True)
    val_df=df[df['file_path'].isin(_va)].reset_index(drop=True)
    test_df=df[df['file_path'].isin(_te)].reset_index(drop=True)
    print(f'[split: B0 CSV 재사용] Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}')
else:
    print('[split: B0 CSV 없음 -> 자체 분할. B0 먼저 실행]')
    train_df, temp_df = train_test_split(df, test_size=cfg.val_ratio+cfg.test_ratio, stratify=df['main_idx'], random_state=cfg.seed)
    _vr=cfg.val_ratio/(cfg.val_ratio+cfg.test_ratio)
    val_df, test_df = train_test_split(temp_df, test_size=1-_vr, stratify=temp_df['main_idx'], random_state=cfg.seed)
    train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)
    print(f'Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}')


## 5. Augmentation — 물리적 타당성 반영

**적용**
- `RandomResizedCrop`: 컨베이어 벨트 다양한 위치/스케일
- `RandomHorizontalFlip`: 좌우 대칭은 의류에 자연스러움
- `ColorJitter`: 선별장 조명 변화
- `GaussianBlur` (확률 낮게): 카메라 흔들림/먼지

**의도적 배제**
- `RandomVerticalFlip` ❌
  - 상하 반전은 옷의 위/아래(목/허리/밑단)를 뒤집어 **종류(category) 분류를 방해**.
    제안서 §3 의 *데이터 이해도* 증명 포인트.


In [ ]:
MEAN = [0.485, 0.456, 0.406]   # ImageNet 통계 (사전학습 정렬)
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((cfg.img_size + 32, cfg.img_size + 32)),
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.5),   # ❌ 의도적 배제 (위 markdown 참조)
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


# === [개선] RandomErasing ===
if cfg.use_random_erasing:
    train_tf.transforms.append(transforms.RandomErasing(p=0.25, scale=(0.02,0.2), value='random'))


## 6. Dataset & DataLoader (+ 클래스 불균형 보정)

In [ ]:
from PIL import Image

class ClothingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['file_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        # [MULTI-TASK] 두번째 라벨도 함께 반환하려면 (img, main_idx, sub_idx) 형태로
        return img, int(row['main_idx'])

train_ds = ClothingDataset(train_df, train_tf)
val_ds   = ClothingDataset(val_df,   eval_tf)
test_ds  = ClothingDataset(test_df,  eval_tf)

# '재사용' 다수 클래스 보정 → 가중 샘플링 + Loss class weight
class_counts = train_df['main_idx'].value_counts().sort_index().values
class_weights = torch.tensor(class_counts.sum() / (len(class_counts) * class_counts),
                             dtype=torch.float32)
sample_weights = train_df['main_idx'].map(
    {i: float(w) for i, w in enumerate(1.0 / class_counts)}
).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

print('클래스 카운트:', dict(zip(cfg.main_classes, class_counts.tolist())))
print('Loss class_weights:', class_weights.tolist())

# === [TRUNCATED IMAGE FIX] DataLoader 워커에도 적용 ===
def _worker_init(worker_id):
    from PIL import ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = True

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                          num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True, worker_init_fn=_worker_init)


## 7. 모델 — ResNet-50 **+ Texture-aware Block**

> B1: stem 뒤에 stride-1 질감 분기를 학습형 α 로 융합. 그 외는 B0 와 동일한
> 단일 헤드. `# [B1]` 주석이 추가/변경 지점입니다.

In [ ]:
# === [B2-MS + 개선] 멀티스케일 질감 분기 + Dropout 헤드 ===
class MultiScaleTextureBlock(nn.Module):
    """미세(3x3) + 넓은영역(dilated 3x3) 병렬 질감 분기 (B2-MS 와 동일)."""
    def __init__(self, mid_ch: int = 32, out_ch: int = 64):
        super().__init__()
        self.fine   = nn.Conv2d(3, mid_ch, 3, stride=1, padding=1, bias=False)
        self.coarse = nn.Conv2d(3, mid_ch, 3, stride=1, padding=2, dilation=2, bias=False)
        self.fuse_bn = nn.BatchNorm2d(mid_ch * 2)
        self.relu = nn.ReLU(inplace=True)
        self.down = nn.Sequential(
            nn.Conv2d(mid_ch * 2, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
    def forward(self, x):
        t = torch.cat([self.fine(x), self.coarse(x)], dim=1)
        t = self.relu(self.fuse_bn(t))
        return self.down(t)


class TextureResNet(nn.Module):
    """[B2-MS + 개선] 멀티스케일 텍스처 + Dropout 헤드.
    구조는 B2-MS, 헤드에 Dropout 추가(개선판 정규화 레시피와 결합)."""
    def __init__(self, num_main: int = 3, use_texture: bool = True,
                 texture_mid_ch: int = 32, dropout_p: float = 0.3, pretrained: bool = True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        self.backbone = resnet50(weights=weights)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.use_texture = use_texture
        if use_texture:
            self.texture_block = MultiScaleTextureBlock(mid_ch=texture_mid_ch, out_ch=64)
            self.texture_alpha = nn.Parameter(torch.zeros(1))
        self.head_main = nn.Sequential(nn.Dropout(dropout_p), nn.Linear(in_feat, num_main))

    def forward(self, x):
        bb = self.backbone
        s = bb.conv1(x); s = bb.bn1(s); s = bb.relu(s); s = bb.maxpool(s)
        if self.use_texture:
            s = s + self.texture_alpha * self.texture_block(x)
        f = bb.layer1(s); f = bb.layer2(f); f = bb.layer3(f); f = bb.layer4(f)
        f = bb.avgpool(f); f = torch.flatten(f, 1)
        return self.head_main(f)


model = TextureResNet(num_main=len(cfg.main_classes),
                      use_texture=cfg.use_texture_block,
                      texture_mid_ch=cfg.texture_mid_ch,
                      dropout_p=cfg.dropout_p).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.2f} M  (MultiScale texture + Dropout head)')


## 8. 학습 루프 (AMP + Cosine + Best-checkpoint)

In [ ]:
import numpy as np
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma=gamma; self.weight=weight
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        ce = F.nll_loss(logp, target, weight=self.weight, reduction='none')
        pt = logp.gather(1, target.unsqueeze(1)).squeeze(1).exp()
        return (((1 - pt) ** self.gamma) * ce).mean()

if cfg.use_focal:
    criterion = FocalLoss(gamma=cfg.focal_gamma, weight=class_weights.to(DEVICE))
    print(f'Loss: Focal(gamma={cfg.focal_gamma}) + class weight')
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=cfg.label_smoothing)
    print('Loss: CrossEntropy + class weight')

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr*0.01)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

def mixup_batch(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total, correct = 0.0, 0, 0
    all_y, all_p = [], []
    for imgs, ys in loader:
        imgs = imgs.to(DEVICE, non_blocking=True); ys = ys.to(DEVICE, non_blocking=True)
        do_mix = train and cfg.use_mixup
        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            if do_mix:
                mixed, ya, yb, lam = mixup_batch(imgs, ys, cfg.mixup_alpha)
                logits = model(mixed)
                loss = lam*criterion(logits, ya) + (1-lam)*criterion(logits, yb)
            else:
                logits = model(imgs); loss = criterion(logits, ys)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss += loss.item()*imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds==ys).sum().item(); total += imgs.size(0)
        all_y += ys.cpu().tolist(); all_p += preds.cpu().tolist()
    p, r, f1, _ = precision_recall_fscore_support(all_y, all_p, average='macro', zero_division=0)
    return total_loss/total, correct/total, p, r, f1

history = []; best_f1, best_state = -1, None
for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _, tr_f1 = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va_loss, va_acc, va_p, va_r, va_f1 = run_epoch(val_loader, train=False)
    scheduler.step()
    history.append(dict(epoch=epoch, tr_loss=tr_loss, va_loss=va_loss, tr_acc=tr_acc, va_acc=va_acc, va_f1=va_f1))
    print(f'[{epoch:02d}/{cfg.epochs}] loss {tr_loss:.3f}/{va_loss:.3f}  acc {tr_acc:.3f}/{va_acc:.3f}  val_F1 {va_f1:.3f}  ({time.time()-t0:.1f}s)')
    if va_f1 > best_f1:
        best_f1 = va_f1; best_state = copy.deepcopy(model.state_dict())
        ckpt = Path(cfg.output_dir) / 'best.pt'
        torch.save({'model': best_state, 'cfg': asdict(cfg), 'val_f1': best_f1, 'epoch': epoch}, ckpt)
print(f'\n[OK] Best val F1-macro: {best_f1:.4f}  (saved -> {ckpt})')
pd.DataFrame(history).to_csv(Path(cfg.output_dir)/'history.csv', index=False)


In [ ]:
if 'hist_df' not in globals():
    hist_df = pd.read_csv(Path(cfg.output_dir) / 'history.csv')

In [ ]:
# 학습 곡선
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_df.epoch, hist_df.tr_loss, label='train'); ax[0].plot(hist_df.epoch, hist_df.va_loss, label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist_df.epoch, hist_df.tr_acc, label='train_acc')
ax[1].plot(hist_df.epoch, hist_df.va_acc, label='val_acc')
ax[1].plot(hist_df.epoch, hist_df.va_f1,  label='val_F1', linestyle='--')
ax[1].set_title('Acc / F1'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'curves.png', dpi=120); plt.show()


## 9. 테스트 평가 — Classification Report & Confusion Matrix

In [ ]:
# best checkpoint 로 복원
model.load_state_dict(best_state)
model.eval()

all_y, all_p, all_prob = [], [], []
with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.use_amp):
    for imgs, ys in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        prob = torch.softmax(logits, 1).cpu().numpy()
        all_prob.append(prob)
        all_p += logits.argmax(1).cpu().tolist()
        all_y += ys.tolist()
all_prob = np.concatenate(all_prob)

print(classification_report(all_y, all_p, target_names=cfg.main_classes, digits=4))
acc = accuracy_score(all_y, all_p)
p, r, f1, _ = precision_recall_fscore_support(all_y, all_p, average='macro', zero_division=0)
pw, rw, f1w, _ = precision_recall_fscore_support(all_y, all_p, average='weighted', zero_division=0)
print(f'Accuracy        : {acc:.4f}')
print(f'Macro P/R/F1    : {p:.4f} / {r:.4f} / {f1:.4f}')
print(f'Weighted P/R/F1 : {pw:.4f} / {rw:.4f} / {f1w:.4f}')

metrics = dict(accuracy=acc, macro_f1=f1, weighted_f1=f1w,
               macro_precision=p, macro_recall=r,
               weighted_precision=pw, weighted_recall=rw)
with open(Path(cfg.output_dir)/'test_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)


In [ ]:
cm = confusion_matrix(all_y, all_p)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[0])
ax[0].set_title('Confusion Matrix (count)')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=cfg.main_classes, yticklabels=cfg.main_classes, ax=ax[1])
ax[1].set_title('Confusion Matrix (row-normalized)')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout(); plt.savefig(Path(cfg.output_dir)/'confusion_matrix.png', dpi=120); plt.show()


In [ ]:
# === [개선] 폐기 임계값 튜닝 ===
from sklearn.metrics import f1_score
if cfg.tune_dispose_threshold:
    model.load_state_dict(best_state); model.eval()
    def _probs(loader):
        P, Y = [], []
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.use_amp):
            for imgs, y in loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                P.append(torch.softmax(model(imgs),1).float().cpu().numpy()); Y += y.tolist()
        return np.concatenate(P), np.array(Y)
    vP, vY = _probs(val_loader); tP, tY = _probs(test_loader)
    DISP = list(cfg.main_classes).index('폐기')
    def pred_thr(P, t):
        pr = P.argmax(1).copy(); pr[P[:, DISP] >= t] = DISP; return pr
    best_t, best_vf1 = 0.5, -1.0
    for t in np.arange(0.10, 0.51, 0.02):
        f1 = f1_score(vY, pred_thr(vP, t), average='macro')
        if f1 > best_vf1: best_vf1, best_t = f1, float(t)
    print(f'선택된 폐기 임계값 t={best_t:.2f}  (val macro-F1 {best_vf1:.4f})\n')
    print('=== 적용 전 (argmax) ===')
    print(classification_report(tY, tP.argmax(1), target_names=cfg.main_classes, digits=4))
    print(f'=== 적용 후 (폐기 t={best_t:.2f}) ===')
    print(classification_report(tY, pred_thr(tP, best_t), target_names=cfg.main_classes, digits=4))
    thr_metrics = dict(threshold=best_t,
                       macro_f1_argmax=float(f1_score(tY, tP.argmax(1), average='macro')),
                       macro_f1_thr=float(f1_score(tY, pred_thr(tP, best_t), average='macro')))
    with open(Path(cfg.output_dir)/'threshold_metrics.json','w',encoding='utf-8') as f:
        json.dump(thr_metrics, f, ensure_ascii=False, indent=2)
    print('\nSaved ->', Path(cfg.output_dir)/'threshold_metrics.json')


## 10. 효율성 지표 — Params / FLOPs / FPS

제안서 §3 의 *현장 적용 가능성(12GB VRAM)* 평가 포인트.
이후 단계 모델과 동일한 입력 크기/배치로 비교해야 의미가 있습니다.

In [ ]:
from thop import profile

# Params
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# FLOPs (입력 1장 기준)
dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=DEVICE)
with torch.no_grad():
    flops, _ = profile(model, inputs=(dummy,), verbose=False)

# FPS (배치=1, 워밍업 후 측정)
model.eval()
with torch.no_grad():
    for _ in range(10): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    N = 100
    for _ in range(N): _ = model(dummy)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    fps = N / (time.time() - t0)

# Peak GPU memory (MB)
mem_mb = (torch.cuda.max_memory_allocated() / 1024**2) if DEVICE.type == 'cuda' else 0.0

eff = dict(params_M=round(n_params/1e6, 3),
           trainable_M=round(n_trainable/1e6, 3),
           gflops=round(flops/1e9, 3),
           fps_batch1=round(fps, 2),
           peak_gpu_mem_MB=round(mem_mb, 1))
print(json.dumps(eff, indent=2))
with open(Path(cfg.output_dir)/'efficiency.json', 'w') as f:
    json.dump(eff, f, indent=2)


## 11. 다음 단계 (Ablation 로드맵)

| Step | 변경점 | 가설 | 측정 지표 |
|---|---|---|---|
| ✅ B0 | Plain ResNet-50 + 단일 헤드 | 기준점 | F1-macro, Params, FLOPs, FPS |
| ✅ **B1** *(본 노트북)* | + Texture-aware Block (stride-1 front conv) | 미세 질감 보존 → '재활용/폐기' 구분 ↑ | (재활용↔폐기) 오분류 감소 |
| B2 | + Multi-Task Head (Main 3-cls + Sub N-cls) | 종류 정보가 상태 분류의 보조 신호 | Main F1 ↑ + Sub F1 동시 보고 |
| B3 | Backbone → DenseNet-121 | Feature reuse | F1-macro ↑, Params ↓ |
| B4 | + Grad-CAM / WSOL (IoU) | XAI: 오염 부위 활성화 | 정성 시각화 + IoU |

**B0 와 비교법**: `baseline_resnet50/test_metrics.json` 의 `macro_f1` 대비
본 노트북 `baseline_B1_texture/test_metrics.json` 의 `macro_f1` 증감, 그리고
Confusion Matrix 의 재활용↔폐기 off-diagonal 변화를 본다.

저장 파일: `best.pt`, `history.csv`, `curves.png`, `test_metrics.json`,
`confusion_matrix.png`, `efficiency.json` (모두 `cfg.output_dir` 아래).
